# Structured Credit — Problem Set 4 Solutions (CDO Loss Distribution & Tranche Pricing)

**Scope.** A portfolio of $N = 20$ credits (bonds/loans), each with a **unit notional**
and **zero recovery** — so if credit $i$ defaults it contributes exactly $1$ unit of loss.
Credit $i$ defaults with probability $p_i$ (given in `StructuredCredit_PSet4.xlsx`), and
**defaults are independent**.

Let $L$ be the total number of losses (defaults) in the portfolio. We need:

| Q | Quantity | Rounding |
|---|---|---|
| 1 | $p^N(3) = \mathbb{P}(L = 3)$ | 3 dp |
| 2 | $\mathbb{E}[L]$ | 2 dp |
| 3 | $\operatorname{Var}(L)$ | 2 dp |
| 4 | Expected tranche loss, $[\ell,u] = [0,2]$ (equity) | 2 dp |
| 5 | Expected tranche loss, $[\ell,u] = [2,4]$ (mezzanine) | 2 dp |
| 6 | Expected tranche loss, $[\ell,u] = [4,20]$ (senior) | 2 dp |

**Notation (as in the lectures).** $p^n(i)$ = probability that **exactly $i$** of the
**first $n$** credits have defaulted. The answer to Q1 is $p^{N}(3)$ with $N=20$.

In [1]:
import warnings
from decimal import Decimal, ROUND_HALF_UP

import numpy as np
import pandas as pd

def round_n(x: float, n: int) -> float:
    # Round-half-up to n decimals (matches the quiz's rounding instructions).
    q = Decimal(1).scaleb(-n)
    return float(Decimal(repr(float(x))).quantize(q, rounding=ROUND_HALF_UP))

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")  # benign xlsx-extension warning

answers = {}

## Data — individual default probabilities

Read straight from the workbook (sheet `LossDistributionCalculations`, column C).

In [2]:
df = pd.read_excel("StructuredCredit_PSet4.xlsx", sheet_name="LossDistributionCalculations",
                   header=1, usecols="B:C")
df.columns = ["credit", "p"]
df["credit"] = range(1, len(df) + 1)   # column B holds Excel formulas (=B4+1); renumber
p = df["p"].to_numpy(dtype=float)
N = len(p)

print(f"N = {N} credits")
df.set_index("credit").T

N = 20 credits


credit,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
p,0.2,0.2,0.06,0.3,0.4,0.65,0.3,0.23,0.02,0.12,0.134,0.21,0.08,0.1,0.1,0.02,0.3,0.015,0.2,0.03


## Building the loss distribution — the recursive (iterative) algorithm

Because the credits have **different** default probabilities, $L$ is **not** binomial
(it is a *Poisson-binomial* random variable). We build its distribution one credit at a time.

**Base case** (no credits yet):

$$
p^0(0) = 1, \qquad p^0(i) = 0 \text{ for } i \ge 1 .
$$

**Recursion.** Add credit $n+1$. "Exactly $i$ defaults among the first $n+1$" happens in two
mutually exclusive ways:

* $i$ defaults among the first $n$ **and** credit $n+1$ survives, or
* $i-1$ defaults among the first $n$ **and** credit $n+1$ defaults.

By independence,

$$
\boxed{\,p^{n+1}(i) = (1-p_{n+1})\,p^{n}(i) + p_{n+1}\,p^{n}(i-1)\,}
\qquad i = 0,1,\dots,n+1,
$$

with the conventions $p^{n}(-1)=0$ and $p^{n}(n+1)=0$. In particular
$p^{n+1}(0) = \prod_{j=1}^{n+1}(1-p_j)$ and $p^{n+1}(n+1)=\prod_{j=1}^{n+1}p_j$.

After $N$ steps, $p^{N}(i) = \mathbb{P}(L=i)$ for $i = 0,\dots,N$. This is exactly the
table you would build in Excel: one column per credit, one row per number of losses.

In [3]:
def loss_distribution(p):
    # Returns the full table P[n, i] = p^n(i) for n = 0..N, i = 0..N.
    N = len(p)
    P = np.zeros((N + 1, N + 1))
    P[0, 0] = 1.0                                   # base case p^0(0) = 1
    for n in range(N):                              # add credit n+1 (prob p[n])
        q = p[n]
        P[n + 1, 0] = (1 - q) * P[n, 0]
        for i in range(1, n + 2):
            P[n + 1, i] = (1 - q) * P[n, i] + q * P[n, i - 1]
    return P

P = loss_distribution(p)
pN = P[N]                                           # final distribution p^N(i)
losses = np.arange(N + 1)

print(f"Sum of probabilities = {pN.sum():.12f}  (must be 1)")
dist = pd.DataFrame({"i (losses)": losses, "p^N(i)": pN, "P(L <= i)": pN.cumsum()})
dist.style.format({"p^N(i)": "{:.6f}", "P(L <= i)": "{:.6f}"}).hide(axis="index")

Sum of probabilities = 1.000000000000  (must be 1)


i (losses),p^N(i),P(L <= i)
0,0.010989,0.010989
1,0.064562,0.075552
2,0.162883,0.238434
3,0.239909,0.478343
4,0.234010,0.712353
5,0.161664,0.874017
6,0.082335,0.956352
7,0.031696,0.988049
8,0.009367,0.997416
9,0.002144,0.999560


The full recursion table $p^n(i)$ (rows = number of credits added $n$, columns = number
of losses $i$), shown for $i \le 8$ — this is the Excel grid in compact form:

In [4]:
pd.DataFrame(P[:, :9], index=pd.Index(range(N + 1), name="n"),
             columns=pd.Index(range(9), name="i")).style.format("{:.5f}")

i,0,1,2,3,4,5,6,7,8
n,,,,,,,,,
0,1.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
1,0.80000,0.20000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
2,0.64000,0.32000,0.04000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
3,0.60160,0.33920,0.05680,0.00240,0.00000,0.00000,0.00000,0.00000,0.00000
4,0.42112,0.41792,0.14152,0.01872,0.00072,0.00000,0.00000,0.00000,0.00000
5,0.25267,0.41920,0.25208,0.06784,0.00792,0.00029,0.00000,0.00000,0.00000
6,0.08844,0.31096,0.36071,0.18760,0.04687,0.00525,0.00019,0.00000,0.00000
7,0.06190,0.24420,0.34578,0.23953,0.08909,0.01773,0.00171,0.00006,0.00000
8,0.04767,0.20227,0.32242,0.26397,0.12369,0.03415,0.00539,0.00044,0.00001


## Question 1 — $p^N(3)$

Read it off the last row of the recursion:

$$
p^{N}(3) = \mathbb{P}(L = 3), \qquad N = 20 .
$$

In [5]:
q1 = pN[3]
answers["Q1"] = round_n(q1, 3)
print(f"p^N(3) = {q1:.6f}  ->  answer: {answers['Q1']:.3f}")

p^N(3) = 0.239909  ->  answer: 0.240


## Question 2 — Expected number of losses

**From the distribution:**

$$
\mathbb{E}[L] = \sum_{i=0}^{N} i\,p^N(i)
$$

**Simple way (linearity of expectation).** Write the total loss as a sum of default indicators,

$$
L = \sum_{j=1}^{N} D_j, \qquad
D_j = \begin{cases} 1 & \text{if credit } j \text{ defaults} \\ 0 & \text{otherwise} \end{cases}
$$

Each $D_j$ is a Bernoulli random variable with parameter $p_j$, so $\mathbb{E}[D_j] = p_j$ and

$$
\boxed{\mathbb{E}[L] = \sum_{j=1}^{N} p_j}
$$

This holds **regardless of correlation**, so it is a free check on the recursion.

In [6]:
EL_dist = losses @ pN
EL_simple = p.sum()
assert np.isclose(EL_dist, EL_simple)

answers["Q2"] = round_n(EL_dist, 2)
print(f"E[L] from distribution   = {EL_dist:.6f}")
print(f"E[L] = sum of p_j        = {EL_simple:.6f}   (check OK)")
print(f"answer: {answers['Q2']:.2f}")

E[L] from distribution   = 3.669000
E[L] = sum of p_j        = 3.669000   (check OK)
answer: 3.67


## Question 3 — Variance of the number of losses

**From the distribution:**

$$
\operatorname{Var}(L) = \mathbb{E}[L^2] - \big(\mathbb{E}[L]\big)^2
= \sum_{i=0}^{N} i^2\,p^N(i) - \Big(\sum_{i=0}^{N} i\,p^N(i)\Big)^2 .
$$

**Check (uses independence).** With independent defaults all covariances vanish, so

$$
\operatorname{Var}(L) = \sum_{j=1}^{N}\operatorname{Var}(D_j) = \sum_{j=1}^{N} p_j(1-p_j).
$$

(Unlike the mean, this shortcut would **fail** if defaults were correlated.)

In [7]:
EL2 = (losses**2) @ pN
VarL_dist = EL2 - EL_dist**2
VarL_simple = (p * (1 - p)).sum()
assert np.isclose(VarL_dist, VarL_simple)

answers["Q3"] = round_n(VarL_dist, 2)
print(f"E[L^2]                   = {EL2:.6f}")
print(f"Var(L) from distribution = {VarL_dist:.6f}")
print(f"Var(L) = sum p_j(1-p_j)  = {VarL_simple:.6f}   (check OK)")
print(f"answer: {answers['Q3']:.2f}")

E[L^2]                   = 15.996780
Var(L) from distribution = 2.535219
Var(L) = sum p_j(1-p_j)  = 2.535219   (check OK)
answer: 2.54


## Tranche losses — formula

A tranche with lower and upper attachment points $\ell < u$ absorbs portfolio losses
between $\ell$ and $u$. Its loss for a realised portfolio loss $L$ is

$$
\text{TL}_{[\ell,u]}(L) = \min\big\lbrace \max(L-\ell,\,0),\; u-\ell\big\rbrace 
= (L-\ell)^+ - (L-u)^+ .
$$

So the tranche is like a **call spread** on the portfolio loss: it loses nothing until $L>\ell$,
then loses one-for-one, and is wiped out (loss capped at its size $u-\ell$) once $L \ge u$.

The **expected tranche loss** uses the distribution we just built:

$$
\boxed{\;\mathbb{E}\big[\text{TL}_{[\ell,u]}\big] = \sum_{i=0}^{N} \min\big\lbrace \max(i-\ell,0),\,u-\ell\big\rbrace \; p^N(i)\;}
$$

**Consistency check.** The tranches $[0,2]$, $[2,4]$, $[4,20]$ are contiguous and cover the
whole capital structure $[0,N]$, so for every outcome $L$ their losses add up to $L$. Hence

$$
\mathbb{E}[\text{TL}_{[0,2]}] + \mathbb{E}[\text{TL}_{[2,4]}] + \mathbb{E}[\text{TL}_{[4,20]}] = \mathbb{E}[L].
$$

In [8]:
def tranche_loss(L, lo, hi):
    return np.minimum(np.maximum(L - lo, 0), hi - lo)

def expected_tranche_loss(pN, lo, hi):
    L = np.arange(len(pN))
    return tranche_loss(L, lo, hi) @ pN

# per-outcome view: how each tranche is hit for each number of losses
pd.DataFrame({
    "i": losses, "p^N(i)": pN,
    "TL[0,2]": tranche_loss(losses, 0, 2),
    "TL[2,4]": tranche_loss(losses, 2, 4),
    "TL[4,20]": tranche_loss(losses, 4, 20),
}).head(10).style.format({"p^N(i)": "{:.6f}"}).hide(axis="index")

i,p^N(i),"TL[0,2]","TL[2,4]","TL[4,20]"
0,0.010989,0,0,0
1,0.064562,1,0,0
2,0.162883,2,0,0
3,0.239909,2,1,0
4,0.234010,2,2,0
5,0.161664,2,2,1
6,0.082335,2,2,2
7,0.031696,2,2,3
8,0.009367,2,2,4
9,0.002144,2,2,5


## Question 4 — Equity tranche $[0, 2]$

$$
\mathbb{E}[\text{TL}_{[0,2]}] = \sum_{i} \min(i, 2)\,p^N(i)
= 0\cdot p^N(0) + 1\cdot p^N(1) + 2\,\big(1 - p^N(0) - p^N(1)\big).
$$

In [9]:
q4 = expected_tranche_loss(pN, 0, 2)
q4_closed = 1 * pN[1] + 2 * (1 - pN[0] - pN[1])
assert np.isclose(q4, q4_closed)

answers["Q4"] = round_n(q4, 2)
print(f"E[TL(0,2)] = {q4:.6f}  ->  answer: {answers['Q4']:.2f}")

E[TL(0,2)] = 1.913459  ->  answer: 1.91


## Question 5 — Mezzanine tranche $[2, 4]$

$$
\mathbb{E}[\text{TL}_{[2,4]}] = \sum_{i} \min\big(\max(i-2,0),\,2\big)\,p^N(i)
= 1\cdot p^N(3) + 2\,\big(1 - \textstyle\sum_{i=0}^{3} p^N(i)\big).
$$

In [10]:
q5 = expected_tranche_loss(pN, 2, 4)
q5_closed = 1 * pN[3] + 2 * (1 - pN[:4].sum())
assert np.isclose(q5, q5_closed)

answers["Q5"] = round_n(q5, 2)
print(f"E[TL(2,4)] = {q5:.6f}  ->  answer: {answers['Q5']:.2f}")

E[TL(2,4)] = 1.283223  ->  answer: 1.28


## Question 6 — Senior tranche $[4, 20]$

Since $u = 20 = N$ the cap never binds, so this is just $\mathbb{E}[(L-4)^+]$:

$$
\mathbb{E}[\text{TL}_{[4,20]}] = \sum_{i=5}^{20} (i-4)\,p^N(i).
$$

Then verify it is consistent with Q2: Q4 + Q5 + Q6 $= \mathbb{E}[L]$.

In [11]:
q6 = expected_tranche_loss(pN, 4, 20)
answers["Q6"] = round_n(q6, 2)
print(f"E[TL(4,20)] = {q6:.6f}  ->  answer: {answers['Q6']:.2f}")

total = q4 + q5 + q6
assert np.isclose(total, EL_dist)
print(f"\nConsistency: {q4:.6f} + {q5:.6f} + {q6:.6f} = {total:.6f} = E[L] = {EL_dist:.6f}  (check OK)")

E[TL(4,20)] = 0.472318  ->  answer: 0.47

Consistency: 1.913459 + 1.283223 + 0.472318 = 3.669000 = E[L] = 3.669000  (check OK)


**Intuition.** The equity tranche $[0,2]$ is only $2/20 = 10\%$ of the notional but carries
about half of the expected portfolio loss — with $\mathbb{E}[L]\approx 3.67$, it is almost surely
wiped out. The senior tranche $[4,20]$ is $80\%$ of the notional yet bears only a small
fraction of expected losses. That concentration of risk in the bottom tranches is the whole
point of tranching.

## Cross-check against the lecture slides (`StructuredCredit.pdf`)

**1. Loss distribution — "Computing the Portfolio Loss Distribution" (Gaussian Copula section).**
The slides compute $p^N(l,t \mid M)$ with an *iterative procedure*, because conditional on the
common factor $M$ the defaults are independent with probabilities

$$
q_i(t \mid M) = \Phi\!\left(\frac{\bar{x}_i(t) - a_i M}{\sqrt{1-a_i^2}}\right).
$$

In this problem set the defaults are **independent** (all factor loadings $a_i = 0$), so
$q_i(t \mid M) = \Phi(\bar{x}_i) = p_i$ for every $M$. The integral over $M$ is then trivial, and the
procedure is exactly the recursion we used:

$$
p^{k+1}(l) = (1-p_{k+1})\,p^{k}(l) + p_{k+1}\,p^{k}(l-1), \qquad p^0(0) = 1 .
$$

**2. Tranche loss function — "The Mechanics of a Synthetic CDO Tranche".** The slides define

$$
\text{TL}^{L,U}(l) = \max\big\lbrace \min\lbrace l\,A(1-R),\; U\rbrace - L,\; 0 \big\rbrace ,
\qquad
\mathbb{E}^{\mathbb{Q}}_0\big[\text{TL}^{L,U}\big] = \sum_{l=0}^{N} \text{TL}^{L,U}(l)\, p(l).
$$

Here each credit has notional $A = 1$ and recovery $R = 0$, so $l\,A(1-R) = l$. This is the same
function as our $\min\lbrace \max(l-L,0),\, U-L\rbrace$ — we check it numerically below.

**3. "A Simple Example: Part I" style.** The slides write the equity-tranche loss as
"tranche size $\times$ P(wiped out) $+ \sum k \times$ P($k$ defaults in the tranche)". For our tranches:

$$
\mathbb{E}[\text{TL}^{0,2}] = 2\,\mathbb{P}(L \ge 2) + 1\cdot\mathbb{P}(L = 1),
\qquad
\mathbb{E}[\text{TL}^{2,4}] = 2\,\mathbb{P}(L \ge 4) + 1\cdot\mathbb{P}(L = 3).
$$

**4. "Some Important Observations".** The slides note that the total expected loss over all tranches
(the expected loss on the index) does not depend on $\rho$ — *"this is not an accident!"*. It equals
$\mathbb{E}[L] = \sum_i p_i$, which is exactly the Q4 + Q5 + Q6 = Q2 check above.

**5. Brute force.** With $N = 20$ there are only $2^{20} \approx 10^6$ default scenarios, so we can also
enumerate every scenario exactly, with no recursion at all, and confirm $p^N(l)$.

In [12]:
l = np.arange(N + 1)

# (2) slide tranche-loss function, A = 1, R = 0
def tl_slide(l, L, U, A=1.0, R=0.0):
    return np.maximum(np.minimum(l * A * (1 - R), U) - L, 0)

for L_, U_ in [(0, 2), (2, 4), (4, 20)]:
    assert np.allclose(tl_slide(l, L_, U_), tranche_loss(l, L_, U_))
print("Slide TL^{L,U}(l) = our tranche_loss(l) for all l and all three tranches   (check OK)")

# (3) "Simple Example" form
q4_slide = 2 * pN[2:].sum() + 1 * pN[1]
q5_slide = 2 * pN[4:].sum() + 1 * pN[3]
assert np.isclose(q4_slide, q4) and np.isclose(q5_slide, q5)
print(f"Simple-Example form:  E[TL 0,2] = {q4_slide:.6f},  E[TL 2,4] = {q5_slide:.6f}   (check OK)")

# (5) brute-force enumeration of all 2^N default scenarios
bits = (np.arange(2**N)[:, None] >> np.arange(N)) & 1           # row = one scenario
scen_prob = np.prod(np.where(bits == 1, p, 1 - p), axis=1)       # independence
pN_brute = np.bincount(bits.sum(axis=1), weights=scen_prob, minlength=N + 1)
assert np.allclose(pN_brute, pN)
print(f"Brute force over {2**N:,} scenarios: p^N(3) = {pN_brute[3]:.6f}  (recursion: {pN[3]:.6f})   (check OK)")

Slide TL^{L,U}(l) = our tranche_loss(l) for all l and all three tranches   (check OK)
Simple-Example form:  E[TL 0,2] = 1.913459,  E[TL 2,4] = 1.283223   (check OK)
Brute force over 1,048,576 scenarios: p^N(3) = 0.239909  (recursion: 0.239909)   (check OK)
